# 05 — Local hot spots: Getis–Ord Gi* and local Moran's I (LISA)

**Question.** Which census tracts sit in a significant concentration of recorded bulletins 2022–2025
(Gi* hot spots) or of low counts (cold spots), and which tracts are local clusters (HH, LL) or outliers
(HL, LH) of the count (LISA)? How many such tracts survive a false-discovery-rate correction, and how much
does this depend on the neighbourhood (queen / rook / KNN-5)?

**Methods and libraries.**

* **Gi\*** (Getis & Ord 1992; Ord & Getis 1995): `esda.G_Local(y, w, transform="B", star=True, permutations=0)`.
  esda's analytic `Zs` uses the Getis–Ord (1992) variance, which is exact only for **binary** weights, so the
  weights are binary and `star=True` adds a self-weight of 1 (the unit counts like each of its neighbours).
  esda's `p_norm` is **one-sided** (`norm.sf(|Zs|)`); the p used here is the two-sided
  $p = 2\,\Phi(-|Z_s|)$ (`scipy.stats.norm.sf`), as in ArcGIS Pro "How Hot Spot Analysis (Getis-Ord Gi*) works".
  Hot = significant with $Z_s>0$, cold = significant with $Z_s<0$.
* **LISA** (Anselin 1995): `esda.Moran_Local(y, w, transformation="r", permutations=499, seed=SEED)` —
  conditional permutation, 499 permutations (Esri default), esda's folded pseudo p, quadrant labels from
  `.q` (1 HH, 2 LH, 3 LL, 4 HL).
* **Multiple testing:** Benjamini–Hochberg FDR at α = 0.05 over all units of one run
  (`statsmodels.stats.multitest.multipletests(method="fdr_bh")`; Benjamini & Hochberg 1995). The raw
  (unadjusted) p < 0.05 counts are shown beside the FDR counts.
* **Weights:** libpysal `Queen`, `Rook`, `KNN(k=5)` on centroids. Islands would be dropped from the run and left
  outside the BH family; no tract is an island under these weights.

**Known limitations reported below.** (i) *LISA permutation floor*: with 499 permutations the smallest pseudo p
is 1/500 = 0.002; BH at 0.05 over n units rejects nothing unless at least 0.002·n/0.05 = n/25 units sit at the
floor (104 of the 2,592 tracts). (ii) *esda zero ties*: esda counts permutation ties with the observed
statistic in one tail only, so units whose own count and whole neighbourhood are zero can come out as
significant LL; such units are counted. (iii) The analytic Gi* normal approximation is poor for skewed
sparse counts and rarely calls cold spots.

**Citations.** Getis, A. & Ord, J.K. (1992). *Geographical Analysis* 24(3). Ord, J.K. & Getis, A. (1995).
*Geographical Analysis* 27(4). Anselin, L. (1995). Local indicators of spatial association — LISA.
*Geographical Analysis* 27(2). Benjamini, Y. & Hochberg, Y. (1995). *JRSS B* 57(1).

In [ ]:
import time
import warnings

warnings.filterwarnings("ignore", message=".*island.*")
warnings.filterwarnings("ignore", message=".*not fully connected.*")
warnings.filterwarnings("ignore", category=FutureWarning)
import esda
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from libpysal.weights import KNN, Queen, Rook, W, w_subset
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

from crime.core.config import NOTEBOOK_OUTPUTS_DIR
from crime.data import SPATIAL_GROUPS as GROUPS
from crime.data import CrimeQuery, Warehouse

import _mapstyle as ms  # notebooks/_mapstyle.py: basemap, overlay, graticule, scale bar, legends

SEED = 20221
ALPHA = 0.05
LISA_PERMUTATIONS = 499
K_NN = 5
SUPPORTS = ("tract",)
WEIGHTS = ("queen", "rook", "knn5")
LAB = {"tract": "census tracts"}
OUT = NOTEBOOK_OUTPUTS_DIR / "05_local_gi_lisa"
OUT.mkdir(parents=True, exist_ok=True)
T0 = time.time()
ms.setup()

In [ ]:
# per-tract counts 2022-2025 from the warehouse, in tract-id order (unit_id = census tract id)
wh = Warehouse()
units = {"tract": wh.census_tracts().rename(columns={"census_tract_id": "unit_id"})}
counts = {("tract", g): wh.counts(CrimeQuery(crime_group=g))["count"].to_numpy(dtype=float) for g in GROUPS}

def build_w(u, kind):
    if kind == "queen":
        w = Queen.from_dataframe(u, ids="unit_id", use_index=False)
    elif kind == "rook":
        w = Rook.from_dataframe(u, ids="unit_id", use_index=False)
    else:
        cent = gpd.GeoDataFrame({"unit_id": u["unit_id"].to_numpy()}, geometry=u.geometry.centroid.to_numpy(), crs=u.crs)
        w = KNN.from_dataframe(cent, k=K_NN, ids="unit_id")
    assert list(w.id_order) == list(u["unit_id"])
    isl = list(w.islands)
    mask = ~u["unit_id"].isin(isl).to_numpy()
    if isl:
        w = w_subset(w, list(u["unit_id"][mask]))
    return w, mask, isl

weights = {}
for s in SUPPORTS:
    for k in WEIGHTS:
        weights[s, k] = build_w(units[s], k)
        print(f"{s:5s} {k:5s}: islands dropped = {weights[s, k][2]}")

## One run = one group × weights

For each run: Gi* labels (two-sided analytic p, raw and BH), LISA labels (pseudo p, raw and BH),
the LISA floor diagnostics, and the number of significant cold / LL units whose own count **and** every
neighbour's count is zero ("all-zero neighbourhood"). Units dropped as islands get the label `island`.

In [ ]:
QUAD = {1: "HH", 2: "LH", 3: "LL", 4: "HL"}

def run(s, g, k):
    w, mask, isl = weights[s, k]
    y = counts[s, g][mask]
    n = len(y)
    # all-zero neighbourhood: own count 0 and every neighbour 0
    nb_sum = np.array([y[[w.id2i[j] for j in w.neighbors[i]]].sum() for i in w.id_order])
    allzero = (y == 0) & (nb_sum == 0)
    # Gi*: binary weights + self-weight 1, analytic z, two-sided p, BH
    wb = W(w.neighbors, id_order=w.id_order, silence_warnings=True)
    gi = esda.G_Local(y, wb, transform="B", star=True, permutations=0)
    z = np.asarray(gi.Zs)
    p_gi = 2 * norm.sf(np.abs(z))
    q_gi = multipletests(p_gi, alpha=ALPHA, method="fdr_bh")[1]
    # LISA: row-standardised weights, 499 conditional permutations, BH on the folded pseudo p
    wr = W(w.neighbors, id_order=w.id_order, silence_warnings=True); wr.transform = "r"
    li = esda.Moran_Local(y, wr, transformation="r", permutations=LISA_PERMUTATIONS, seed=SEED, keep_simulations=False)
    p_li = li.p_sim
    q_li = multipletests(p_li, alpha=ALPHA, method="fdr_bh")[1]
    quad = np.vectorize(QUAD.get)(li.q)
    def gilab(pv):
        return np.where(pv <= ALPHA, np.where(z > 0, "hot", "cold"), "ns")
    def lilab(pv):
        return np.where(pv <= ALPHA, quad, "ns")
    def giconf(pv):   # Esri-style confidence bins of the (raw or BH-adjusted) two-sided p
        lvl = np.select([pv <= 0.01, pv <= 0.05, pv <= 0.10], ["99%", "95%", "90%"], "")
        return np.where(lvl != "", np.where(z > 0, "hot spot ", "cold spot ") + lvl, "ns")
    lab = pd.DataFrame({"unit_id": w.id_order, "count": y, "gi_z": z, "gi_p2": p_gi, "gi_q_bh": q_gi,
                        "gi_raw": gilab(p_gi), "gi_fdr": gilab(q_gi),
                        "gi_conf_raw": giconf(p_gi), "gi_conf_fdr": giconf(q_gi), "lisa_I": li.Is, "lisa_p": p_li,
                        "lisa_q_bh": q_li, "lisa_quadrant": quad, "lisa_raw": lilab(p_li), "lisa_fdr": lilab(q_li),
                        "allzero_neighbourhood": allzero})
    floor = 1 / (LISA_PERMUTATIONS + 1)
    row = dict(support=s, group=g, weights=k, n=n, n_islands=len(isl), events=int(y.sum()),
               n_allzero_neighbourhood=int(allzero.sum()))
    for c in ("hot", "cold"):
        row[f"gi_{c}_raw"] = int((lab.gi_raw == c).sum()); row[f"gi_{c}_fdr"] = int((lab.gi_fdr == c).sum())
    row["gi_cold_fdr_allzero"] = int(((lab.gi_fdr == "cold") & allzero).sum())
    for c in ("HH", "LL", "HL", "LH"):
        row[f"lisa_{c}_raw"] = int((lab.lisa_raw == c).sum()); row[f"lisa_{c}_fdr"] = int((lab.lisa_fdr == c).sum())
    row["lisa_LL_raw_allzero"] = int(((lab.lisa_raw == "LL") & allzero).sum())
    row["lisa_LL_fdr_allzero"] = int(((lab.lisa_fdr == "LL") & allzero).sum())
    row["lisa_min_p"] = p_li.min()
    row["lisa_n_at_floor"] = int(np.isclose(p_li, floor).sum())
    row["lisa_floor_units_needed_for_bh"] = int(np.ceil(floor * n / ALPHA))
    return row, lab

rows, labels = [], {}
for s in SUPPORTS:
    for g in GROUPS:
        for k in WEIGHTS:
            r, lab = run(s, g, k)
            rows.append(r); labels[s, g, k] = lab
tab = pd.DataFrame(rows)
tab.to_csv(OUT / "local_counts_all_weights.csv", index=False)
print(f"{len(rows)} runs in {time.time() - T0:.0f} s")

In [ ]:
# per-unit labels (queen) for every group and support
for s in SUPPORTS:
    frames = []
    for g in GROUPS:
        frames.append(labels[s, g, "queen"].assign(group=g))
    pd.concat(frames).to_csv(OUT / f"labels_{s}_queen.csv", index=False)
cols = ["support", "group", "weights", "n", "gi_hot_raw", "gi_hot_fdr", "gi_cold_raw", "gi_cold_fdr", "gi_cold_fdr_allzero"]
print("Gi* hot / cold spots: raw two-sided p < 0.05 vs BH-FDR q < 0.05 (number of units)")
tab[cols]

In [ ]:
cols = ["support", "group", "weights", "n", "lisa_HH_raw", "lisa_HH_fdr", "lisa_LL_raw", "lisa_LL_fdr",
        "lisa_LL_raw_allzero", "lisa_LL_fdr_allzero", "lisa_HL_raw", "lisa_HL_fdr", "lisa_LH_raw", "lisa_LH_fdr",
        "lisa_min_p", "lisa_n_at_floor", "lisa_floor_units_needed_for_bh"]
print("LISA clusters / outliers: raw pseudo p < 0.05 vs BH-FDR (number of units); permutation floor diagnostics")
tab[cols]

## Before and after the FDR correction: all groups, queen weights

Each figure: **left before FDR** (unadjusted p), **right after Benjamini–Hochberg FDR** (BH-adjusted q, one
family = all units of the run). Every unit is drawn; *not significant* is light grey; basemap Esri World Light Gray Canvas (contextily).

* **Gi\*** — colour = hot (reds) / cold (blues), shade **and** hatch = confidence level as in ArcGIS Pro's Gi_Bin
  (99 %: p ≤ 0.01, solid; 95 %: p ≤ 0.05, `//`; 90 %: p ≤ 0.10, `..`). The hot / cold counts used elsewhere in this
  notebook (α = 0.05) are the 99 % + 95 % classes.
* **LISA** — HH red, LL blue (clusters, no hatch); outliers hatched: HL orange `//`, LH light blue `\\`; p ≤ 0.05.

The legends give the number of units before → after FDR.


In [ ]:
GI_STYLE = {"hot spot 99%": ("#b2182b", ""), "hot spot 95%": ("#ef6548", "//"), "hot spot 90%": ("#fdbb84", ".."),
            "cold spot 99%": ("#2166ac", ""), "cold spot 95%": ("#67a9cf", "//"), "cold spot 90%": ("#d1e5f0", ".."),
            "ns": ("#d9d9d9", ""), "island": ("white", "")}
GI_GROUPS = {"Hot spots": ["hot spot 99%", "hot spot 95%", "hot spot 90%"],
             "Cold spots": ["cold spot 99%", "cold spot 95%", "cold spot 90%"], "Other": ["ns", "island"]}
LI_STYLE = {"HH": ("#d7191c", ""), "LL": ("#2c7bb6", ""), "HL": ("#fdae61", "//"), "LH": ("#abd9e9", "\\\\"),
            "ns": ("#d9d9d9", ""), "island": ("white", "")}
LI_GROUPS = {"Clusters": ["HH", "LL"], "Outliers": ["HL", "LH"], "Other": ["ns", "island"]}
NAMES = {"HH": "HH cluster (high among high)", "LL": "LL cluster (low among low)", "HL": "HL outlier (high among low)",
         "LH": "LH outlier (low among high)", "ns": "not significant", "island": "island (no neighbours, not tested)"}


def labelled(s, g, k, col):
    lab = labels[s, g, k].set_index("unit_id")[col]
    return units[s]["unit_id"].map(lab).fillna("island")


def draw(ax, s, g, k, col, style, title):
    tract = s == "tract"
    ms.plot_categories(ax, units[s], labelled(s, g, k, col), style, hatch_repeat=2 if tract else 1,
                       linewidth=0.05 if tract else 0.3, edgecolor="white" if tract else "0.45")
    ms.decorate(ax, units[s].crs)
    ax.set_title(title, fontsize=9)


def legend_groups(groups, style, keep):
    out = {name: [(NAMES.get(c, c), *style[c]) for c in cs if c in keep] for name, cs in groups.items()}
    return {k: v for k, v in out.items() if v}


def before_after(s, g, stat, raw_col, fdr_col, style, groups, what):
    fig, axes = plt.subplots(1, 2, figsize=(13, 7))
    raw, fdr = labelled(s, g, "queen", raw_col), labelled(s, g, "queen", fdr_col)
    for ax, col, lab, thr in ((axes[0], raw_col, raw, f"before FDR: unadjusted {what} p"),
                              (axes[1], fdr_col, fdr, "after BH-FDR: q")):
        n_sig = int((~lab.isin(["ns", "island"]) & ~lab.str.endswith("90%")).sum())
        draw(ax, s, g, "queen", col, style, f"{thr} ≤ 0.05 — {n_sig} significant {LAB[s]}")
    vr, vf = raw.value_counts(), fdr.value_counts()
    keep = [c for c in style if vr.get(c, 0) or vf.get(c, 0) or c == "ns"]
    counts = {NAMES.get(c, c): f"{vr.get(c, 0)} → {vf.get(c, 0)}" for c in keep}
    ms.category_legend(fig, legend_groups(groups, style, keep), counts=counts, bbox=(0.5, 0.06), fontsize=7.5,
                       title="units before → after FDR")
    fig.suptitle(f"{stat}, bulletins 2022–2025, all groups, {LAB[s]} (n = {len(raw):,}), queen weights", fontsize=10)
    fig.tight_layout(rect=(0, 0.06, 1, 0.97))
    name = "gi" if raw_col.startswith("gi") else "lisa"
    fig.savefig(OUT / f"map_{name}_all_{s}_before_after_fdr.png", dpi=300, bbox_inches="tight")
    plt.show(); plt.close(fig)


for s in SUPPORTS:
    before_after(s, "all", "Getis–Ord Gi* (binary weights + self, two-sided analytic p)", "gi_conf_raw", "gi_conf_fdr",
                 GI_STYLE, GI_GROUPS, "two-sided")
for s in SUPPORTS:
    before_after(s, "all", "Local Moran's I (row-standardised, 499 permutations)", "lisa_raw", "lisa_fdr",
                 LI_STYLE, LI_GROUPS, "pseudo")


In [ ]:
# counts table: category x before / after FDR, per support (queen)
HOT95, COLD95 = ["hot spot 99%", "hot spot 95%"], ["cold spot 99%", "cold spot 95%"]

def before_after_counts(g, k="queen"):
    rows = {}
    for s in SUPPORTS:
        lab = labels[s, g, k]
        for when, gi_col, li_col in (("before FDR", "gi_conf_raw", "lisa_raw"), ("after FDR", "gi_conf_fdr", "lisa_fdr")):
            gi_vc, li_vc = lab[gi_col].value_counts(), lab[li_col].value_counts()
            col = {f"Gi* {c}": int(gi_vc.get(c, 0)) for c in list(GI_STYLE)[:6] + ["ns"]}
            col["Gi* hot spot (α = 0.05: 99 % + 95 %)"] = int(lab[gi_col].isin(HOT95).sum())
            col["Gi* cold spot (α = 0.05: 99 % + 95 %)"] = int(lab[gi_col].isin(COLD95).sum())
            col |= {f"LISA {c}": int(li_vc.get(c, 0)) for c in ("HH", "LL", "HL", "LH", "ns")}
            rows[(s, when)] = col
    t = pd.DataFrame(rows)
    t.columns.names = ["support", "labels"]
    return t

ba_all = before_after_counts("all")
# consistency with the run table (gi_raw / gi_fdr use the same p <= 0.05 rule)
r = tab.query("group == 'all' and weights == 'queen'").set_index("support")
for s in SUPPORTS:
    assert ba_all.loc["Gi* hot spot (α = 0.05: 99 % + 95 %)", (s, "before FDR")] == r.loc[s, "gi_hot_raw"]
    assert ba_all.loc["Gi* hot spot (α = 0.05: 99 % + 95 %)", (s, "after FDR")] == r.loc[s, "gi_hot_fdr"]
ba_all.to_csv(OUT / "before_after_fdr_counts_all.csv")
print("Units per category, all groups, queen: before FDR (unadjusted p) vs after BH-FDR (q); Gi* by confidence bin")
ba_all


In [ ]:
# the other groups: the same before / after counts (queen), one table
OTHER = [g for g in GROUPS if g != "all"]
ba_other = pd.concat({g: before_after_counts(g) for g in OTHER}, axis=1, names=["group"])
ba_other.to_csv(OUT / "before_after_fdr_counts_other_groups.csv")
ba_other.loc[["Gi* hot spot (α = 0.05: 99 % + 95 %)", "Gi* cold spot (α = 0.05: 99 % + 95 %)",
              "LISA HH", "LISA LL", "LISA HL", "LISA LH"]]


## Small multiples — the other groups (queen, after BH-FDR)

Before / after counts for these groups are in the table above.

In [ ]:
for s in SUPPORTS:
    fig, axes = plt.subplots(2, len(OTHER), figsize=(4 * len(OTHER), 8.8))
    for j, g in enumerate(OTHER):
        r = tab.query("support == @s and group == @g and weights == 'queen'").iloc[0]
        draw(axes[0, j], s, g, "queen", "gi_conf_fdr", GI_STYLE, f"Gi* {g}: {r.gi_hot_fdr} hot, {r.gi_cold_fdr} cold (q ≤ 0.05)")
        draw(axes[1, j], s, g, "queen", "lisa_fdr", LI_STYLE,
             f"LISA {g}: HH {r.lisa_HH_fdr}, LL {r.lisa_LL_fdr},\nHL {r.lisa_HL_fdr}, LH {r.lisa_LH_fdr}")
    for ax in axes.flat:
        ax.tick_params(labelsize=5)
    groups = {"Gi* hot spots (BH q)": [(c, *GI_STYLE[c]) for c in GI_GROUPS["Hot spots"]],
              "Gi* cold spots (BH q)": [(c, *GI_STYLE[c]) for c in GI_GROUPS["Cold spots"]],
              "LISA (q ≤ 0.05)": [(NAMES[c], *LI_STYLE[c]) for c in ("HH", "LL", "HL", "LH")],
              "Other": [("not significant", *GI_STYLE["ns"])]}
    ms.category_legend(fig, groups, bbox=(0.5, 0.06), fontsize=8)
    fig.suptitle(f"Bulletins 2022–2025 per unit, {LAB[s]}, queen weights, after BH-FDR "
                 f"(top: Gi* two-sided analytic p, confidence bin of the BH q; bottom: LISA, 499 permutations)", fontsize=11)
    fig.tight_layout(rect=(0, 0.07, 1, 0.96)); fig.savefig(OUT / f"small_multiples_{s}_queen.png", dpi=300, bbox_inches="tight")
    plt.show(); plt.close(fig)
print(f"runtime {time.time() - T0:.0f} s")


In [ ]:
# numbers quoted in the Findings cell
show = ["support", "group", "weights", "gi_hot_raw", "gi_hot_fdr", "gi_cold_raw", "gi_cold_fdr", "lisa_HH_raw", "lisa_HH_fdr",
        "lisa_LL_raw", "lisa_LL_fdr", "lisa_LL_raw_allzero", "lisa_n_at_floor", "lisa_floor_units_needed_for_bh", "n_allzero_neighbourhood"]
print(tab[show].to_string(index=False))

## Results pending corrected execution

Regenerate this section from the corrected output tables. Historical narrative has been removed because it disagreed with executed output or preceded changes to inference and missing-data handling.
